In [5]:
import os   
import json
import numpy as np
from dotenv import load_dotenv
from typing import Dict, List, TypedDict, Literal    
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class ConversationState(TypedDict):
    messages: List[Dict[str, str]]          
    current_message: str                    
    selected_task: str                      
    task_message_relevance: float           
    generated_questions: List[str]          
    selected_question: str                 
    question_sample_simliarity: List[float] 
    question_message_relevance: float      
    user_answer_score: float                     
    conversation_mode: Literal["assessment", "casual"] 


@dataclass
class ChatbotConfig:
    openai_api_key: str                     
    assessment_threshold: float = 0.5       # 검사 대화 vs 일상 대화 진입 기준
    fallback_threshold: float = 0.7         # 질문 재생성 vs 출력 진입 기준
    model_name: str = "gpt-4o-mini"         


ASSESSMENT_TASKS = {
    "registration_recall": {
        "description": 
        """기억 등록은 즉각적인 기억력을 평가하고, 회상은 기억을 유지하는 능력을 평가하는 항목입니다.
          messages 최근 5개 turn 내에서 동일 선상에서 비교될 수 있는 단어/고유명사가 3개 이상 나오면 본 평가내역을 활용할 수 있습니다.""",
        "example_questions": [
            "아까 말씀하신 과일 중 사과, 배, 포도를 어릴 때 가장 좋아했던 순서대로 말씀해주세요.",
            "아까 말씀하신 자녀 중 영희, 철수, 길동이를 살가운 순서대로 말씀해주시겠어요?",
            "아까 말씀하신 공책, 필통, 샤프를 어릴 적 갖고 싶었던 순서대로 말씀해주세요.",
            "콩, 생선, 고추들을 어릴 적 싫어했던 순서대로 말씀해주세요.",
            "콩, 생선, 고추들을 요즘 좋아하시는 순서대로 말씀해주세요."
        ]
    },
    "Naming": {
        "description": 
        """표시된 사물의 이름을 기억해내는 능력을 평가합니다. 
        사진데이터에서 위치관계가 명확한 사물이 있을 경우 본 평가 항목을 사용하기 적당합니다.""",
        "example_questions": [
            "사진 속 어린아이가 들고있는 물체를 뭐라고 불러요?",
            "손가락에 끼고 있는 것의 이름은 뭔가요?",
            "친구가 가지고 놀고 있는 물건의 이름은 뭐에요?",
            "사진 속 할머니 옆에 있는 꽃의 이름은 뭔가요?",
            "아이가 안고 있는 동물의 이름은 뭐에요?",
            "케이크 밑에 있는 가구 이름은 뭔가요?"
        ]
    },
    "time_orientation": {
        "description": 
        """현재 자신이 놓여있는 시간, 날짜, 계절 등의 상황을 올바르게 인식하는 능력을 평가합니다.
        시간 관련 humanmassage가 본 평가항목에 대한 트리거가 됩니다.
        example_questions의 응용을 최소화하여 질문을 생성하세요.
        """,
        "example_questions": [
            "올해는 몇년도인가요?"
        ]
    }
}


load_dotenv()

API_KEY = os.getenv("GPT_API_KEY")

config = ChatbotConfig(
    openai_api_key=API_KEY,  
    assessment_threshold=0.8,
    fallback_threshold=0.7
)


class CompleteMiniDementiaChatbot:


    def __init__(self, config: ChatbotConfig):
        self.config = config
        self.llm = ChatOpenAI(
            model=config.model_name,
            openai_api_key=config.openai_api_key,
            temperature=0.3
        )
        self.vectorizer = TfidfVectorizer(stop_words='english')
        print(f"테스트 챗봇 초기화 완료 (모델: {config.model_name})")
    
    def complete_chat_flow(self, message: str) -> Dict:
        print(f"사용자 입력 메세지: {message}")
        
        # 1단계: 사용자 메세지의 태스크 별 적합도 계산
        print("1단계: 태스크 별 적합도 계산...")
        task_scores = {}
        
        for task_name, task_info in ASSESSMENT_TASKS.items():
            prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            평가 영역: {task_name} - {description}
            
            사용자 메시지가 이 평가 영역과 얼마나 관련이 있는지 0-1 사이의 점수로 평가해주세요.
            0: 전혀 관련 없음, 1: 매우 관련 있음
            
            점수만 반환해주세요 (예: 0.8):
            """)
            
            try:
                response = self.llm.invoke(prompt.format_messages(
                    message=message,
                    task_name=task_name,
                    description=task_info["description"]
                ))
                score = float(response.content.strip())
                task_scores[task_name] = score
                print(f"  {task_name}: {score:.2f}")
            except:
                task_scores[task_name] = 0.0
        
        selected_task = max(task_scores.items(), key=lambda x: x[1])
        print(f"선택된 태스크: {selected_task[0]} (점수: {selected_task[1]:.2f})")
        
        # 2단계: 적합도 threshold 체크 (assessment/casual 대화 결정)
        print(f"\n2단계: 적합도 threshold 체크...")
        print(f"선택된 태스크 적합도: {selected_task[1]:.2f}")
        print(f"Assessment 진입 임계값: {self.config.assessment_threshold}")
        
        # 태스크 적합도가 임계값을 넘는지 확인
        if selected_task[1] < self.config.assessment_threshold:
            print(f"일상 대화 모드 (적합도 {selected_task[1]:.2f} < 임계값 {self.config.assessment_threshold})")
            
            casual_prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            
            다음 메시지에 대해 자연스럽고 친근한 일상 대화로 응답해주세요.
            
            응답:
            """)
            
            try:
                response = self.llm.invoke(casual_prompt.format_messages(message=message))
                ai_response = response.content.strip()
                response_type = "casual"
            except:
                ai_response = "응답 생성에 실패했습니다."
                response_type = "error"
            
            return {
                "user_message": message,
                "selected_task": selected_task[0],
                "task_message_relevance": selected_task[1],
                "generated_questions": [],
                "selected_question": "",
                "question_message_relevance": 0.0,
                "response_type": response_type,
                "ai_response": ai_response,
                "workflow_stage": "casual_chat"
            }
        
        print(f"평가 모드 진입 (적합도 {selected_task[1]:.2f} >= 임계값 {self.config.assessment_threshold})")
        task_info = ASSESSMENT_TASKS[selected_task[0]]
        
        # 3-1: 질문 생성 (5개)
        question_gen_prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{message}"
        평가 영역: {task_name}
        예시 질문들: {examples}
        
        위 사용자 메시지의 맥락을 고려하여, {task_name} 평가를 위한 자연스러운 질문 5개를 생성해주세요.
        예시 질문들을 참고하되, 사용자의 메시지와 자연스럽게 이어지도록 만들어주세요.
        
        각 질문을 새 줄로 구분하여 번호 없이 나열해주세요:
        """)
        
        try:
            response = self.llm.invoke(question_gen_prompt.format_messages(
                message=message,
                task_name=selected_task[0],
                examples="\n".join(task_info["example_questions"])
            ))
            
            generated_questions = [q.strip() for q in response.content.split('\n') if q.strip()]
            for i, q in enumerate(generated_questions, 1):
                print(f"  {i}. {q}")
        except:
            generated_questions = []
            print("질문 생성 실패")
        
        if not generated_questions:
            ai_response = "평가 질문 생성에 실패했습니다."
            response_type = "error"
            return {
                "user_message": message,
                "selected_task": selected_task[0],
                "task_message_relevance": selected_task[1],
                "generated_questions": generated_questions,
                "selected_question": "",
                "question_message_relevance": 0.0,
                "response_type": response_type,
                "ai_response": ai_response,
                "workflow_stage": "question_generation_failed"
            }
        
        # 3-2: 예시 질문과 생성 질문을 비교해 유사도 기반 최적 질문 선택
        print("\n3-2단계: 최적 질문 선택...")
        example_questions = task_info["example_questions"]
        all_questions = generated_questions + example_questions
        
        try:
            tfidf_matrix = self.vectorizer.fit_transform(all_questions)
            generated_vectors = tfidf_matrix[:len(generated_questions)]
            example_vectors = tfidf_matrix[len(generated_questions):]
            
            similarity_matrix = cosine_similarity(generated_vectors, example_vectors)
            max_similarities = np.max(similarity_matrix, axis=1)
            
            best_question_idx = np.argmax(max_similarities)
            selected_question = generated_questions[best_question_idx]
            
            print(f"질문별 유사도 점수:")
            for i, (q, score) in enumerate(zip(generated_questions, max_similarities)):
                marker = "선택됨" if i == best_question_idx else "      "
                print(f"  {marker} {score:.3f}: {q}")
                
        except:
            selected_question = generated_questions[0]
            print("유사도 계산 실패, 첫 번째 질문 선택")
        
        # 4단계: 질문-메시지 맥락 적합성 검증
        print(f"\n4단계: 질문-메시지 맥락 적합성 검증...")
        question_context_prompt = ChatPromptTemplate.from_template("""
        사용자 메시지: "{current_message}"
        생성된 질문: "{selected_question}"
        
        이 질문이 사용자 메시지와 자연스럽게 이어지는 대화인지 0-1 사이의 점수로 평가해주세요.
        평가 기준:
        - 기존 대화의 자연스러운 흐름
        - 맥락의 연결성  
        - 갑작스럽지 않은 전환
        - 노인 사용자가 답변할 수 있는 적절한 질문인지
        
        점수만 반환해주세요 (예: 0.8):
        """)
        
        try:
            response = self.llm.invoke(question_context_prompt.format_messages(
                current_message=message,
                selected_question=selected_question
            ))
            question_message_relevance = float(response.content.strip())
            print(f"질문-메시지 맥락 점수: { question_message_relevance:.2f}")
        except:
            question_message_relevance = 0.0
        
        # 최종 결과 결정
        if question_message_relevance >= self.config.fallback_threshold:
            ai_response = selected_question
            response_type = "assessment"
            workflow_stage = "assessment_question_output"
            print(f"최종 평가 질문 출력")
        else:
            print(f"질문이 맥락에 적합하지 않음 - 일상 대화로 전환")
            
            fallback_prompt = ChatPromptTemplate.from_template("""
            사용자 메시지: "{message}"
            
            위 메시지에 대해 자연스럽고 친근한 응답을 하면서, 
            인지 능력을 간접적으로 평가할 수 있는 질문을 포함해주세요.
            
            응답:
            """)
            
            try:
                response = self.llm.invoke(fallback_prompt.format_messages(message=message))
                ai_response = response.content.strip()
                response_type = "fallback"
                workflow_stage = "fallback_conversation"
            except:
                ai_response = "응답 생성에 실패했습니다."
                response_type = "error"
                workflow_stage = "error"
        
        result = {
            "user_message": message,
            "selected_task": selected_task[0],
            "task_message_relevance": selected_task[1],
            "generated_questions": generated_questions,
            "selected_question": selected_question,
            "question_message_relevance": question_message_relevance,
            "response_type": response_type,
            "ai_response": ai_response,
            "workflow_stage": workflow_stage
        }
        
        print(f"\nAI 응답: {ai_response}")
        print(f"응답 타입: {response_type}")
        
        return result

# 테스트 실행
if __name__ == "__main__":
    # 미니 챗봇 생성
    mini_chatbot = CompleteMiniDementiaChatbot(config)
    
    # 테스트 시나리오
    test_scenarios = [
        "걔는 내 큰 딸이야.",
        "사과, 배, 바나나 외에도 과일은 모두 귀했지.",
        "그 때는 다들 그렇게 지냈어."
    ]
    
    # 각 시나리오 테스트
    results = []
    for scenario in test_scenarios:
        print(f"\n{'테스트 시나리오':=^60}")
        result = mini_chatbot.complete_chat_flow(scenario)
        results.append(result)
    
    # 결과 요약
    print(f"\n{'테스트 결과 요약':=^60}")
    for i, result in enumerate(results, 1):
        print(f"\n{i}. 메시지: '{result['user_message']}'")
        print(f"   선택 태스크: {result['selected_task']} (점수: {result['task_message_relevance']:.2f})")
        print(f"   질문 맥락 점수: {result['question_message_relevance']:.2f}")
        print(f"   응답 타입: {result['response_type']}")
        print(f"   워크플로우 단계: {result['workflow_stage']}")
        print(f"   AI 응답: {result['ai_response'][:50]}...")

테스트 챗봇 초기화 완료 (모델: gpt-4o-mini)

==========================테스트 시나리오==========================
사용자 입력 메세지: 걔는 내 큰 딸이야.
1단계: 태스크 별 적합도 계산...
  registration_recall: 1.00
  Naming: 0.00
  time_orientation: 0.00
선택된 태스크: registration_recall (점수: 1.00)

2단계: 적합도 threshold 체크...
선택된 태스크 적합도: 1.00
Assessment 진입 임계값: 0.8
평가 모드 진입 (적합도 1.00 >= 임계값 0.8)
  1. 아까 말씀하신 자녀 중 큰 딸, 작은 딸, 아들을 살가운 순서대로 말씀해주시겠어요?
  2. 큰 딸이 어릴 때 가장 좋아했던 장난감이나 놀이를 말씀해주실 수 있나요?
  3. 큰 딸과 함께했던 특별한 추억이나 경험을 말씀해주실 수 있을까요?
  4. 아까 언급하신 자녀들 중에서 가장 자주 함께 시간을 보냈던 순서대로 말씀해주시겠어요?
  5. 큰 딸이 좋아했던 음식이나 간식을 어릴 적 순서대로 말씀해주실 수 있나요?

3-2단계: 최적 질문 선택...
질문별 유사도 점수:
  선택됨 0.547: 아까 말씀하신 자녀 중 큰 딸, 작은 딸, 아들을 살가운 순서대로 말씀해주시겠어요?
         0.257: 큰 딸이 어릴 때 가장 좋아했던 장난감이나 놀이를 말씀해주실 수 있나요?
         0.000: 큰 딸과 함께했던 특별한 추억이나 경험을 말씀해주실 수 있을까요?
         0.155: 아까 언급하신 자녀들 중에서 가장 자주 함께 시간을 보냈던 순서대로 말씀해주시겠어요?
         0.201: 큰 딸이 좋아했던 음식이나 간식을 어릴 적 순서대로 말씀해주실 수 있나요?

4단계: 질문-메시지 맥락 적합성 검증...
질문-메시지 맥락 점수: 0.60
질문이 맥락에 적합하지 않음 - 일상 대화로 전환

AI 응답: "아, 큰 딸이구나